# CURATOR on Kaggle GPUs
Notebook settings (right panel): **Accelerator: GPU T4 x2** for every step (not TPU: the stack needs an NVIDIA GPU), **Internet: on**, **Input: your CURATOR dataset**, **Secrets: HF_TOKEN switched on**.

Every session: set `STEP` in cell 5, then **Run All** (or **Save Version -> Save & Run All** to run in the background). A session that ends early is resumed by running the same `STEP` again. Full instructions: `README_KAGGLE.md` in the package.

In [ ]:
%%bash
# 1) Copy the CURATOR package into a writable folder. Works whether Kaggle kept the uploaded zip
#    or extracted it (it usually extracts .zip files on upload).
set -e
IN=${KAGGLE_INPUT:-/kaggle/input}
mkdir -p /kaggle/working/curator && cd /kaggle/working/curator
ZIP=$(find "$IN" -maxdepth 6 -name 'curator-gpu-kaggle.zip' 2>/dev/null | head -1)
INFO=$(find "$IN" -maxdepth 6 -name 'BUNDLE_INFO.json' 2>/dev/null | head -1)
if [ -n "$ZIP" ]; then
  echo "unzipping $ZIP"; unzip -q -o "$ZIP"
elif [ -n "$INFO" ]; then
  echo "copying the extracted package from $(dirname "$INFO")"; cp -r "$(dirname "$INFO")"/. .
else
  echo "ERROR: the CURATOR dataset is not attached (right panel -> Input -> Add Input)"; exit 1
fi
chmod -R u+w .
echo "package:"; cat BUNDLE_INFO.json; echo
ls
nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv || echo "NO GPU: set Accelerator to GPU T4 x2"


In [ ]:
# 2) Where runs are mirrored while they train. Put YOUR Hugging Face username here.
#    Needs the Kaggle secret HF_TOKEN (Add-ons -> Secrets; a Hugging Face token with WRITE access),
#    switched on for this notebook. SYNC = "" keeps runs only in /kaggle/working (then: Save Version!).
import os

SYNC = "hf://YOUR_HF_USERNAME/curator-runs"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as exc:
    print("No HF_TOKEN secret available:", exc)
    SYNC = ""
if "YOUR_HF_USERNAME" in SYNC:
    print("Edit SYNC above with your Hugging Face username.")
    SYNC = ""
SYNC_ARG = f"--sync-to {SYNC}" if SYNC else ""
print("sync:", SYNC or "OFF - use Save Version before the session ends, or the runs are lost")


In [ ]:
# 3) Install the training stack for this GPU (T4: vLLM in its own virtualenv; P100: HF backend). ~5-10 min.
%cd /kaggle/working/curator
!python run.py setup


In [ ]:
# 4) First session only: ~10 min end-to-end check on the GPU (3 rounds, calibration, checkpoint,
#    restore from the synced copy, resume). With SYNC set it also proves the Hugging Face upload works.
RUN_SMOKE = True          # set to False after it has passed once
if RUN_SMOKE:
    !python run.py smoke {SYNC_ARG}


In [ ]:
# 5) The run for THIS session. Pick one STEP (see README_KAGGLE.md, "What to run, in order").
#    Rerunning the same STEP in a new session RESUMES it from the last checkpoint; finished runs are skipped.
STEP = "compare"          # compare | sysid | ablation | seeds12 | loo
COMMANDS = {
    "compare":  "compare --methods curator,uniform,lp --seeds 0",            # GPU T4 x2, ~3.5 h
    "sysid":    "sysid --calib-items 50 --final-items 100 --seeds 0,1",       # GPU T4 x2, ~2 h (2 seeds in parallel)
    "ablation": "compare --methods curator_nocal,pilot_uniform --seeds 0",    # GPU T4 x2, ~2 h
    "seeds12":  "compare --methods curator,uniform,lp --seeds 1,2",           # GPU T4 x2, ~5.5 h
    "loo":      "loo --methods uniform --seeds 0",                            # GPU T4 x2, ~5.5 h
}
print("running:", COMMANDS[STEP], SYNC_ARG)
!python run.py {COMMANDS[STEP]} {SYNC_ARG}


In [ ]:
# 6) Where every run is (safe to run any time, also while step 5 runs in another tab)
!python run.py status


In [ ]:
# 7) Results: rebuilt automatically after every finished run, every 20 min and at the end of step 5.
#    Download /kaggle/working/results/curator_results.zip from the Output panel (right side ->
#    Output -> results -> curator_results.zip -> download); with SYNC it is also in your HF dataset under results/.
from pathlib import Path

!python run.py report --quiet {SYNC_ARG}
res = Path("/kaggle/working/results")
cmp = res / "comparison.md"
print(cmp.read_text() if cmp.exists() else "no comparison yet (no run has finished)")
print("files:", sorted(str(f.relative_to(res)) for f in res.rglob("*") if f.is_file())[:40])
z = res / "curator_results.zip"
print("download:", z, f"({z.stat().st_size / 1e6:.1f} MB)" if z.exists() else "(not written yet)")


In [ ]:
# 8) Before "Save Version": delete the vLLM virtualenv (several GB) so it is not saved as output.
#    The runs, checkpoints and reports in /kaggle/working/curator_runs are kept.
!rm -rf /kaggle/working/venv_vllm /kaggle/working/curator/.curator_python
!du -sh /kaggle/working/curator_runs 2>/dev/null
